In [ ]:
from flax import nnx
from functools import partial
from flax.typing import PaddingLike
from jax import eval_shape, ShapeDtypeStruct, numpy as jnp
from math import prod

### Simple RNN Classifier

In [ ]:
class RNNClassifier(nnx.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_classes, *, rngs):
        self.embedding = nnx.Embed(
            num_embeddings=vocab_size,
            features=embed_dim,
            rngs=rngs,
        )

        cell = nnx.SimpleCell(
            in_features=embed_dim,
            hidden_features=hidden_dim,
            rngs=rngs,
        )

        self.rnn = nnx.RNN(cell, return_carry=True)
        self.fc = nnx.Linear(hidden_dim, n_classes, rngs=rngs)

    def __call__(self, x, lengths=None):
        # x: [B, T]
        x = self.embedding(x)          # [B, T, E]

        # h_last: [B, H]
        # outputs: [B, T, H]
        h_last, outputs = self.rnn(x, seq_lengths=lengths)

        return self.fc(h_last)         # [B, n_classes]

### LSTM Classifier

In [ ]:
class LSTMClassifier(nnx.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_classes, *, rngs):
        self.embedding = nnx.Embed(
            num_embeddings=vocab_size,
            features=embed_dim,
            rngs=rngs,
        )

        cell = nnx.OptimizedLSTMCell(
            in_features=embed_dim,
            hidden_features=hidden_dim,
            rngs=rngs,
        )

        self.lstm = nnx.RNN(cell, return_carry=True)
        self.fc = nnx.Linear(hidden_dim, n_classes, rngs=rngs)

    def __call__(self, x, lengths=None):
        x = self.embedding(x)          # [B, T, E]

        # carry = (c_last, h_last)
        (c_last, h_last), outputs = self.lstm(x, seq_lengths=lengths)

        return self.fc(h_last)         # [B, n_classes]

### Bidirectional RNN

In [ ]:
## Simple

class BiRNNClassifier(nnx.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_classes, *, rngs):
        self.embedding = nnx.Embed(
            num_embeddings=vocab_size,
            features=embed_dim,
            rngs=rngs,
        )

        forward = nnx.RNN(
            nnx.SimpleCell(embed_dim, hidden_dim, rngs=rngs),
            return_carry=True,
        )

        backward = nnx.RNN(
            nnx.SimpleCell(embed_dim, hidden_dim, rngs=rngs),
            return_carry=True,
        )

        self.birnn = nnx.Bidirectional(
            forward_rnn=forward,
            backward_rnn=backward,
            return_carry=True,
        )

        self.fc = nnx.Linear(2 * hidden_dim, n_classes, rngs=rngs)

    def __call__(self, x, lengths=None):
        x = self.embedding(x)          # [B, T, E]

        # carry = (forward_carry, backward_carry)
        # each carry is [B, H] for SimpleCell
        (h_fwd, h_bwd), outputs = self.birnn(x, seq_lengths=lengths)

        h = jnp.concatenate([h_fwd, h_bwd], axis=-1)  # [B, 2H]

        return self.fc(h)
    
## Manual

class ManualBiRNNClassifier(nnx.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim,
        hidden_dim,
        n_classes,
        *,
        rngs,
    ):
        self.embedding = nnx.Embed(
            num_embeddings=vocab_size,
            features=embed_dim,
            rngs=rngs,
        )

        self.rnns = nnx.List()

        # Forward RNN
        fwd_cell = nnx.SimpleCell(
            in_features=embed_dim,
            hidden_features=hidden_dim,
            rngs=rngs,
        )
        self.rnns.append(
            nnx.RNN(fwd_cell, return_carry=True)
        )

        # Backward RNN
        bwd_cell = nnx.SimpleCell(
            in_features=embed_dim,
            hidden_features=hidden_dim,
            rngs=rngs
        )
        self.rnns.append(
            nnx.RNN(bwd_cell, return_carry=True)
        )

        self.fc = nnx.Linear(
            in_features=2 * hidden_dim,
            out_features=n_classes,
            rngs=rngs,
        )

    def __call__(self, x, lengths=None):
        # x: [B, T]
        x = self.embedding(x)  # [B, T, E]

        # Forward pass
        h_fwd, y_fwd = self.rnns[0](
            x,
            seq_lengths=lengths,
        ) # h_fwd: [B, H]; y_fwd: [B, T, H]

        # Backward pass
        h_bwd, y_bwd = self.rnns[1](
            x,
            seq_lengths=lengths,
            reverse=True,
            keep_order=True,
        ) # h_bwd: [B, H]; y_bwd: [B, T, H]
        

        h = jnp.concatenate([h_fwd, h_bwd], axis=-1)  # [B, 2H]

        return self.fc(h)  # [B, n_classes]

### Deep RNN

In [ ]:
class DeepRNNClassifier(nnx.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim,
        hidden_dim,
        n_classes,
        n_layers=2,
        *,
        rngs,
    ):
        self.embedding = nnx.Embed(
            num_embeddings=vocab_size,
            features=embed_dim,
            rngs=rngs,
        )

        self.layers = nnx.List()

        for i in range(n_layers):
            in_dim = embed_dim if i == 0 else hidden_dim

            cell = nnx.SimpleCell(
                in_features=in_dim,
                hidden_features=hidden_dim,
                rngs=rngs,
            )

            self.layers.append(
                nnx.RNN(cell, return_carry=True)
            )

        self.fc = nnx.Linear(hidden_dim, n_classes, rngs=rngs)

    def __call__(self, x, lengths=None):
        x = self.embedding(x)  # [B, T, E]

        h_last = None

        for rnn in self.layers:
            h_last, x = rnn(x, seq_lengths=lengths)
            # x: [B, T, H], passed into next recurrent layer

        return self.fc(h_last)  # type: ignore # [B, n_classes]
    
from jax import numpy as jnp

jnp.ix_

In [ ]:
from jax import numpy as jnp

In [ ]:
jnp.mgrid[:2, :3]

In [ ]:
jnp.mgrid[0:5]

In [ ]:
jnp.mgrid[0:10:2]

In [ ]:
jnp.mgrid[0:4:3j] # 0 to 4 with 3 points evenly spaced

In [ ]:
k = jnp.ogrid[:2, :3] # ogrid returns open/sparse coordinate grids that rely on broadcasting
k, k[0] + k[1]

In [ ]:
rows = jnp.array([0, 2])
cols = jnp.array([1, 3, 4])

jnp.ix_(rows, cols)

In [ ]:
jnp.meshgrid(rows, cols, sparse=True, indexing="ij")

In [ ]:
jnp.ogrid[:3:2, :5]

In [ ]:
jnp.arange(0,10,2)

In [ ]:
jnp.mgrid[0:10:2]

In [ ]:
jnp.linspace(0,1,5)

In [ ]:
jnp.mgrid[0:1:5j]

In [ ]:
jnp.mgrid[0:10:2, 0:20:5]

In [49]:
k = jnp.ones((1,5,3,1,1))
jnp.squeeze(k).shape

(5, 3)

In [50]:
k = jnp.ones((1,5,3,1,1))
jnp.squeeze(k, axis=0).shape

(5, 3, 1, 1)